In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import *
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *
from src.utils.training import *
from src.utils.visualization import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 3.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Setting seed to 42


In [2]:
DEBUG = False
SKIP_TRAINING = False
EXP_NAME = "baseline"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}, SKIP_TRAINING={SKIP_TRAINING}")

device = DEVICE

Starting experiment baseline. DEBUG=False, SKIP_TRAINING=False


In [3]:
resnet = MakeResNet18().to(device)
MODEL_NAME = ""

total_params, model_size_mb = get_model_summary(resnet)
print(f"Total Parameters: {total_params:,}")
print(f"Model Size: {model_size_mb:.2f} MB")


Total Parameters: 11,173,962
Model Size: 42.63 MB


Split data set into train-validation-test.
We are using 80% train, 20% validation split

In [4]:
trainloader, valloader, testloader, train_set, val_set, test_set = get_cifar10_loaders_and_splits()
resnet_optimizer, resnet_scheduler = get_optimizer_and_scheduler(resnet)

100%|██████████| 170M/170M [00:38<00:00, 4.47MB/s]


Using default 80/20% split.
Original train-val size: 50000
Train size: 40000
Val size: 10000
Test size: 10000


In [5]:
if not SKIP_TRAINING:
    train_model(
        model=resnet,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=resnet_optimizer,
        scheduler=resnet_scheduler,
        device=device,
        experiment_name=EXP_NAME,
        model_name=MODEL_NAME,
        DEBUG=DEBUG
    )

Model weights will be saved to: /kaggle/working/artifacts/checkpoints/baseline_.pth
Stats will be saved to: /kaggle/working/artifacts/stats/baseline_.pkl

=== Starting Training:  with 100 epochs ===
Epoch 1/100 | Loss: 1.992 | Val Acc: 36.32%
    --> New Best Saved: 51.79%
Epoch 2/100 | Loss: 1.464 | Val Acc: 51.79%
    --> New Best Saved: 61.73%
Epoch 3/100 | Loss: 1.206 | Val Acc: 61.73%
    --> New Best Saved: 64.67%
Epoch 4/100 | Loss: 1.035 | Val Acc: 64.67%
    --> New Best Saved: 66.80%
Epoch 5/100 | Loss: 0.926 | Val Acc: 66.80%
    --> New Best Saved: 72.24%
Epoch 6/100 | Loss: 0.823 | Val Acc: 72.24%
    --> New Best Saved: 73.27%
Epoch 7/100 | Loss: 0.720 | Val Acc: 73.27%
    --> New Best Saved: 75.41%
Epoch 8/100 | Loss: 0.658 | Val Acc: 75.41%
Epoch 9/100 | Loss: 0.602 | Val Acc: 75.06%
    --> New Best Saved: 75.79%
Epoch 10/100 | Loss: 0.571 | Val Acc: 75.79%
    --> New Best Saved: 79.18%
Epoch 11/100 | Loss: 0.545 | Val Acc: 79.18%
Epoch 12/100 | Loss: 0.522 | Val Acc

In [6]:
debug_suff = "_DEBUG" if DEBUG else ""
load_weights(resnet, experiment_name=EXP_NAME, model_name=(MODEL_NAME+ debug_suff), device=device)
if not SKIP_TRAINING: 
    print(f'Final test accuracy is: {calculate_accuracy(resnet, testloader, device):.3f}')

Final test accuracy is: 88.570
